# Unlimited-OCR-2 - PDF to Markdown (Google Colab)

This notebook turns every PDF in `input/` into Markdown in `output/`, using the
official **Unlimited-OCR-2** model. It creates both folders automatically, each
with a `README.txt` explaining what goes there.

## How to use

1. **Runtime -> Change runtime type -> GPU** (T4 is the minimum; L4/A100 is better).
2. **Runtime -> Run all**.
3. Wait until the notebook prints **WAITING FOR PDFs**.
4. In the left **Files** pane, open `input/` and upload one or more PDFs.
5. The notebook processes them automatically. Results appear in `output/`.

The first run downloads about **7 GB** of model weights and can take 10-20 minutes.

> **Colab storage is temporary.** Files in `/content` disappear when the runtime
> disconnects. Set `USE_DRIVE = True` in the configuration cell to keep `input/`
> and `output/` in Google Drive instead.

> This notebook uses the native **Transformers** backend, so it does not need an
> SGLang server. For production or very long documents, use the project's
> `start_server.sh` + `run_ocr.py` on a dedicated NVIDIA Linux machine.


In [ ]:
#@title 1. Install dependencies (first run only)
# The model's remote code targets transformers 4.52.1. Do not upgrade it.
%pip install -q "transformers==4.52.1" "huggingface-hub==0.36.0" "tokenizers==0.21.4" "accelerate==1.6.0" addict einops easydict "pymupdf==1.27.2.2" pillow sentencepiece

import sys
print("Python:", sys.version.split()[0])


Python: 3.13.16


## Configuration

The defaults are chosen to work on a free Colab T4 (16 GB). On a smaller GPU the
notebook automatically falls back to one page per request; on a bigger GPU it
uses larger chunks.

If you change `transformers` or other packages after this notebook has imported
them, use **Runtime -> Restart runtime** and run again.

Colab may print dependency-resolver warnings about `gradio`, `datasets` or other
preinstalled packages while downgrading `transformers`. They are safe to ignore
for this notebook; only the OCR cell matters.


In [ ]:
#@title 2. Configuration and folders
from pathlib import Path
import os, sys, json, re, time, shutil, gc, importlib.util

# True = keep input/output in Google Drive (survives runtime restarts).
# False = use /content/input and /content/output (faster, but temporary).
USE_DRIVE = False
DRIVE_FOLDER = "/content/drive/MyDrive/UnlimitedOCR"

MODEL_ID = "baidu/Unlimited-OCR-2"

# 200 dpi is plenty for this model; 300 is slower and uses more memory.
DPI = 200

# 0 = all pages. Set e.g. 10 to limit very long PDFs.
MAX_PAGES = 0

# auto = base for multi-page and for low-VRAM GPUs; gundam for one page on >=20 GB
# base = one global view per page (lowest memory)
# gundam = tiled views (more detail, more memory)
MODE = "auto"

# 0 = choose from VRAM at load time. Set 1 for the lowest memory usage.
CHUNK_PAGES = 0

MAX_TOKENS_PER_PAGE = 8000
MAX_PAGE_TOKENS = 20000
MAX_MODEL_LEN = 131072

# Reprocess PDFs whose .md already exists.
FORCE = False

# Keep the rendered page JPEGs in output/.work/ (they are deleted otherwise).
KEEP_PAGES = False

# Download a ZIP of output/ at the end.
AUTO_DOWNLOAD_ZIP = True

# How long the "wait for PDFs" cell waits before giving up.
WAIT_MINUTES = 30

# auto = bfloat16 on Ampere+, float16 on T4/Turing.
# Set "bfloat16" or "float16" to force one.
FORCE_DTYPE = "auto"

# Use the model's fused Triton attention kernels. If you see Triton errors,
# set this to False (slower, but pure PyTorch).
USE_TRITON = True

if MODE not in ("auto", "base", "gundam"):
    raise ValueError("MODE must be auto, base or gundam")
MAX_TOKENS_PER_PAGE = max(1, int(MAX_TOKENS_PER_PAGE))

if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        ROOT = Path(DRIVE_FOLDER)
    except Exception as exc:
        print("Drive mount failed, using temporary storage:", exc)
        ROOT = Path("/content")
else:
    ROOT = Path("/content")

INPUT_DIR = ROOT / "input"
OUTPUT_DIR = ROOT / "output"
CODE_DIR = Path("/content/Unlimited-OCR")
INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_README = """UNLIMITED-OCR-2 - INPUT FOLDER

Put PDF files in this folder (subfolders are allowed).

After uploading, re-run the "Wait for PDFs, then process" cell, or use
Runtime -> Run all. Already-processed PDFs are skipped unless FORCE = True.

The notebook renders each page and sends the document to baidu/Unlimited-OCR-2.
Results are written to ../output/.

Colab storage is temporary. If you want files to survive a runtime restart,
set USE_DRIVE = True in the configuration cell and put PDFs in the Drive
folder instead.
"""

OUTPUT_README = """UNLIMITED-OCR-2 - OUTPUT FOLDER

For each input file "name.pdf" you get:

  name.md          final Markdown (figures are in name.assets/)
  name.raw.txt     raw model output (layouts, tables, boxes)
  name.meta.json   pages, timings, token counts, finish_reason
  name.assets/     cropped figure images referenced by name.md

run_summary.json is written after every batch.

You can re-run the notebook; existing .md files are skipped unless FORCE = True.
"""

(INPUT_DIR / "README.txt").write_text(INPUT_README, encoding="utf-8")
(OUTPUT_DIR / "README.txt").write_text(OUTPUT_README, encoding="utf-8")

print("Input folder :", INPUT_DIR)
print("Output folder:", OUTPUT_DIR)
print("Upload PDFs to the input folder, then run the processing cell.")


Input folder : /content/input
Output folder: /content/output
Upload PDFs to the input folder, then run the processing cell.


In [ ]:
#@title 3. Check the GPU and load the model
import os, sys, gc, time
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("SPARSE_DIAG_LOG", "0")
os.environ.setdefault("CODE_STAMP_LOG", "0")
os.environ.setdefault("RECALL_LOG", "0")

import torch
if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. Use Runtime -> Change runtime type -> GPU, then run again."
    )

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
cap = torch.cuda.get_device_capability(0)
print(f"GPU: {gpu_name} | {vram_gb:.1f} GB | compute {cap[0]}.{cap[1]}")
if vram_gb < 10:
    raise RuntimeError(
        f"Only {vram_gb:.1f} GB VRAM. Unlimited-OCR-2 needs about 12 GB minimum "
        "(the BF16 weights alone are ~6.7 GB). Use a T4/L4/A100 runtime."
    )
if vram_gb < 12:
    print("WARNING: less than 12 GB VRAM; use base mode and one page at a time.")

if FORCE_DTYPE == "bfloat16":
    DTYPE = torch.bfloat16
elif FORCE_DTYPE == "float16":
    DTYPE = torch.float16
else:
    try:
        # Exclude CUDA's software BF16 emulation: on T4/Turing it is very slow.
        native_bf16 = torch.cuda.is_bf16_supported(including_emulation=False)
    except TypeError:
        native_bf16 = torch.cuda.is_bf16_supported()
    DTYPE = torch.bfloat16 if native_bf16 else torch.float16

if DTYPE == torch.float16:
    print("BF16 is not native on this GPU; using float16 instead (T4/Turing).")
else:
    print("Using bfloat16.")

from transformers import AutoConfig, AutoModel, AutoTokenizer
import transformers as _tf

print("transformers:", _tf.__version__)
if _tf.__version__ != "4.52.1":
    raise RuntimeError(
        f"transformers {_tf.__version__} is loaded, but this notebook needs 4.52.1. "
        "Run Runtime -> Restart runtime, then Runtime -> Run all."
    )

print("Loading tokenizer ...")
print("You can upload PDFs to", INPUT_DIR, "while the model loads.")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

print("Loading model (~6.7 GB) ...")
load_kwargs = dict(
    trust_remote_code=True,
    use_safetensors=True,
    torch_dtype=DTYPE,
)
if not USE_TRITON:
    config = AutoConfig.from_pretrained(MODEL_ID, trust_remote_code=True)
    sparse_cfg = getattr(config, "sparse_attention_config", None)
    if isinstance(sparse_cfg, dict):
        sparse_cfg["dash_use_triton"] = False
        print("Triton kernels: off (dense PyTorch fallback)")
    else:
        print("WARNING: could not disable Triton in the model config")
    load_kwargs["config"] = config
else:
    print("Triton kernels: on")
try:
    model = AutoModel.from_pretrained(MODEL_ID, low_cpu_mem_usage=True, **load_kwargs)
except Exception as exc:
    print("low_cpu_mem_usage failed, retrying without it:", exc)
    gc.collect()
    model = AutoModel.from_pretrained(MODEL_ID, **load_kwargs)

model = model.eval().cuda()
torch.set_grad_enabled(False)
print(f"Model loaded. dtype={next(model.parameters()).dtype}")
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


def patch_image_dtype(dtype):
    """The model preprocesses images as bfloat16. Cast them to the model dtype."""
    patched = 0
    wanted = {type(model).__module__}
    try:
        wanted.add(type(model.model).__module__)
    except Exception:
        pass
    for name, module in list(sys.modules.items()):
        if not (name.endswith("modeling_unlimitedocr") or name in wanted):
            continue
        original = getattr(module, "_uocr_build_inputs", None)
        if original is None or getattr(original, "_uocr_dtype_patched", False):
            continue

        def wrapper(*args, __orig=original, __dtype=dtype, **kwargs):
            input_ids, images, seq_mask, spatial = __orig(*args, **kwargs)
            if images is not None:
                images = (images[0].to(__dtype), images[1].to(__dtype))
            return input_ids, images, seq_mask, spatial

        wrapper._uocr_dtype_patched = True
        module._uocr_build_inputs = wrapper
        patched += 1
    return patched


if DTYPE == torch.float16:
    n = patch_image_dtype(torch.float16)
    print(f"Patched image preprocessing for float16: {n} module(s).")
    if n == 0:
        print("WARNING: could not patch image preprocessing; float16 inference may fail.")
        print("         Set FORCE_DTYPE = 'bfloat16' if this GPU supports BF16.")

# Pick a safe number of pages per request for this GPU.
if CHUNK_PAGES > 0:
    CHUNK = CHUNK_PAGES
elif vram_gb >= 40:
    CHUNK = 0          # whole document in one request
elif vram_gb >= 22:
    CHUNK = 8
elif vram_gb >= 15:
    CHUNK = 2
else:
    CHUNK = 1

print("Pages per request:", "whole document" if CHUNK == 0 else CHUNK)


GPU: Tesla T4 | 15.6 GB | compute 7.5
BF16 is not native on this GPU; using float16 instead (T4/Turing).
transformers: 4.52.1
Loading tokenizer ...
You can upload PDFs to /content/input while the model loads.


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/549 [00:00<?, ?B/s]

Loading model (~6.7 GB) ...
Triton kernels: on


... (21 outputs left out)

[PageRope] mode=page2d head_dim=128 tok_theta=10000.0 fallback_1d_theta=500000.0 page_pairs=1 page_theta=256.0 page_mod=2 tok_freq_mode=resample page_span=0
[PageRope] pairs page/1 tok/63 capacity page=3pages tok=27143tokens
[PageRope main] mode=page2d head_dim=128 page_pairs=1 page_theta=256.0 tok_theta=10000.0 tok_freq_mode=resample page_mod=2 capacity page=3pages tok=27143tokens
[DASH] differentiable DASH attention active (entmax router, block_size=32, local_window=128, index_dim=128, main_rope=page2d, idx_rope=page2d, page_bucket=True); first dash layer_idx=0


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Model loaded. dtype=torch.float16
GPU memory used: 6.69 GB
Patched image preprocessing for float16: 1 module(s).
Pages per request: 2


In [ ]:
#@title 4. Download the upstream post-processor
import os
import urllib.request
import importlib.util

POST_DIR = CODE_DIR / "utils"
POST_DIR.mkdir(parents=True, exist_ok=True)
POST_PATH = POST_DIR / "process_unlimitedOCR2.py"
POST_URL = "https://raw.githubusercontent.com/baidu/Unlimited-OCR/main/utils/process_unlimitedOCR2.py"

if not POST_PATH.exists() or POST_PATH.stat().st_size < 1000:
    print("Downloading process_unlimitedOCR2.py ...")
    try:
        urllib.request.urlretrieve(POST_URL, POST_PATH)
    except Exception as exc:
        print("Download failed, trying wget:", exc)
        os.system(f"wget -q '{POST_URL}' -O '{POST_PATH}'")

if not POST_PATH.exists() or POST_PATH.stat().st_size < 1000:
    raise RuntimeError("Could not download process_unlimitedOCR2.py")

spec = importlib.util.spec_from_file_location("process_unlimitedOCR2", POST_PATH)
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)
convert = module.convert

_test = convert("<PAGE><BLOCK>text[0,0,100,100]<FORK>ok")
print("Post-processor ready:", _test)


Post-processor ready: ok


In [ ]:
#@title 5. Helper functions
import gc, time, json, re, shutil
from pathlib import Path

from PIL import Image

try:
    import pymupdf
except ImportError:
    import fitz as pymupdf

RE_IMAGE_REF = re.compile(
    r"!\[\]\(page(\d+)_(-?\d+)_(-?\d+)_(-?\d+)_(-?\d+)\.jpg\)", re.I
)


def safe_doc_id(pdf):
    try:
        rel = pdf.resolve().relative_to(INPUT_DIR.resolve())
    except ValueError:
        rel = Path(pdf.name)
    raw = rel.with_suffix("").as_posix().replace("/", "__")
    doc_id = re.sub(r"[^\w.\-]+", "_", raw).strip("._")
    return doc_id or "document"


def list_pdfs():
    return sorted(
        p for p in INPUT_DIR.rglob("*")
        if p.is_file()
        and p.suffix.lower() == ".pdf"
        and not p.name.startswith("~$")
    )


def render_pdf(pdf, dest):
    doc = pymupdf.open(str(pdf))
    try:
        if doc.needs_pass:
            raise RuntimeError("PDF is password protected")
        total = doc.page_count
        if total == 0:
            raise RuntimeError("PDF has no pages")
        count = min(total, MAX_PAGES) if MAX_PAGES > 0 else total
        dest.mkdir(parents=True, exist_ok=True)
        pages = []
        for i in range(count):
            pix = doc[i].get_pixmap(dpi=DPI, alpha=False)
            path = dest / f"page_{i + 1:04d}.jpg"
            path.write_bytes(pix.tobytes("jpeg", jpg_quality=92))
            pages.append(path)
        return pages, total
    finally:
        doc.close()


def normalize_chunk(raw):
    text = (raw or "").strip()
    while text.endswith("<PAGE>"):
        text = text[: -len("<PAGE>")].rstrip()
    if text and "<PAGE>" not in text:
        text = "<PAGE>" + text
    return text


def crop_figure_refs(markdown, pages, assets_dir):
    if not pages:
        return markdown
    assets_dir.mkdir(parents=True, exist_ok=True)
    made = 0

    def repl(match):
        nonlocal made
        page_no = int(match.group(1))
        if page_no < 1 or page_no > len(pages):
            return ""
        try:
            with Image.open(pages[page_no - 1]) as src:
                img = src.convert("RGB")
        except Exception:
            return ""
        width, height = img.size
        x1, y1, x2, y2 = (
            max(0, min(1000, int(match.group(i)))) for i in range(2, 6)
        )
        px1, py1 = int(x1 / 1000 * width), int(y1 / 1000 * height)
        px2, py2 = int(x2 / 1000 * width), int(y2 / 1000 * height)
        if px2 <= px1 or py2 <= py1:
            return ""
        name = f"page{page_no}_{x1}_{y1}_{x2}_{y2}.jpg"
        try:
            img.crop((px1, py1, px2, py2)).save(assets_dir / name, "JPEG", quality=90)
        except Exception:
            return ""
        made += 1
        return f"![]({assets_dir.name}/{name})"

    result = RE_IMAGE_REF.sub(repl, markdown)
    if made == 0:
        shutil.rmtree(assets_dir, ignore_errors=True)
    return result


def infer_chunk(paths):
    crop_mode = None if MODE == "auto" else (MODE == "gundam")
    # On smaller GPUs, use one global view even for a single page. Gundam tiling
    # on a 16 GB T4 can otherwise push the vision encoder over the memory limit.
    if MODE == "auto" and len(paths) == 1 and vram_gb < 20:
        crop_mode = False
    max_new = min(
        MAX_TOKENS_PER_PAGE * len(paths),
        MAX_MODEL_LEN - 2048,
    )
    return model.infer(
        tokenizer,
        prompt="Multi page parsing.",
        image_file=[str(p) for p in paths],
        crop_mode=crop_mode,
        eval_mode=True,
        tps_interval=500,
        no_repeat_ngram_size=35,
        ngram_window=1024 if len(paths) > 1 else 128,
        page_count_guard=True,
        max_page_tokens=MAX_PAGE_TOKENS,
        max_new_tokens=max(1, max_new),
        max_length=MAX_MODEL_LEN,
        return_dict=True,
    )


def infer_with_retry(paths):
    try:
        return [infer_chunk(paths)]
    except RuntimeError as exc:
        if "out of memory" not in str(exc).lower():
            raise
        torch.cuda.empty_cache()
        gc.collect()
        if len(paths) > 1:
            mid = len(paths) // 2
            print(f"  OOM: splitting {len(paths)} pages -> {mid} + {len(paths) - mid}")
            return infer_with_retry(paths[:mid]) + infer_with_retry(paths[mid:])
        raise RuntimeError(
            "CUDA out of memory on a single page. Try MODE='base', "
            "lower MAX_TOKENS_PER_PAGE, or a larger GPU."
        )


def process_pdf(pdf):
    started = time.time()
    doc_id = safe_doc_id(pdf)
    md_path = OUTPUT_DIR / f"{doc_id}.md"
    raw_path = OUTPUT_DIR / f"{doc_id}.raw.txt"
    meta_path = OUTPUT_DIR / f"{doc_id}.meta.json"
    assets_dir = OUTPUT_DIR / f"{doc_id}.assets"

    if md_path.exists() and not FORCE:
        print(f"  skip (already exists): {md_path.name}")
        return {"doc": doc_id, "status": "skipped"}

    work = OUTPUT_DIR / ".work" / doc_id
    shutil.rmtree(work, ignore_errors=True)
    work.mkdir(parents=True, exist_ok=True)

    pages, total = render_pdf(pdf, work)
    print(f"  rendered {len(pages)}/{total} page(s) at {DPI} dpi")

    chunk_size = CHUNK if CHUNK > 0 else len(pages)
    chunks = [pages[i : i + chunk_size] for i in range(0, len(pages), chunk_size)]

    raw_parts = []
    chunk_meta = []
    truncated = False

    for index, chunk in enumerate(chunks, 1):
        print(f"  chunk {index}/{len(chunks)}: {chunk[0].stem} .. {chunk[-1].stem}")
        print("    running model (this can take several minutes) ...")
        chunk_started = time.time()
        results = infer_with_retry(chunk)
        for res in results:
            raw_parts.append(normalize_chunk(res.get("text", "")))
            chunk_meta.append(
                {
                    "pages": res.get("expected_pages"),
                    "num_pages_out": res.get("num_pages_out"),
                    "completion_tokens": len(res.get("token_ids") or []),
                    "finish_reason": res.get("finish_reason"),
                    "seconds": round(res.get("time") or (time.time() - chunk_started), 1),
                }
            )
            if res.get("finish_reason") == "length":
                truncated = True
        raw_path.write_text("\n".join(p for p in raw_parts if p), encoding="utf-8")
        print(f"    {sum(len(p) for p in raw_parts):,} chars so far")
        torch.cuda.empty_cache()
        gc.collect()

    raw = "\n".join(p for p in raw_parts if p).strip()
    if not raw:
        raise RuntimeError("The model returned an empty result.")
    raw_path.write_text(raw, encoding="utf-8")

    markdown = convert(raw)
    if not markdown.strip():
        raise RuntimeError("Post-processing returned empty Markdown.")
    shutil.rmtree(assets_dir, ignore_errors=True)
    markdown = crop_figure_refs(markdown, pages, assets_dir)
    md_path.write_text(markdown, encoding="utf-8")

    elapsed = round(time.time() - started, 1)
    meta = {
        "doc": doc_id,
        "source": str(pdf),
        "pages_rendered": len(pages),
        "pages_total": total,
        "dpi": DPI,
        "mode": MODE,
        "chunk_pages": CHUNK,
        "chunks": chunk_meta,
        "truncated": truncated,
        "model": MODEL_ID,
        "dtype": str(DTYPE).replace("torch.", ""),
        "seconds": elapsed,
        "markdown_chars": len(markdown),
        "raw_chars": len(raw),
    }
    meta_path.write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")

    if not KEEP_PAGES:
        shutil.rmtree(work, ignore_errors=True)
        try:
            (OUTPUT_DIR / ".work").rmdir()
        except OSError:
            pass

    print(f"  saved {md_path.name} ({len(markdown):,} chars, {elapsed}s)")
    if truncated:
        print("  WARNING: output hit max_new_tokens; the last page may be cut off.")
    return {
        "doc": doc_id,
        "status": "ok",
        "pages": len(pages),
        "seconds": elapsed,
        "truncated": truncated,
    }


def run_all():
    pdfs = list_pdfs()
    if not pdfs:
        print("No PDFs found in", INPUT_DIR)
        return []

    print(f"Found {len(pdfs)} PDF(s)")
    results = []
    batch_started = time.time()
    for index, pdf in enumerate(pdfs, 1):
        rel = pdf.relative_to(INPUT_DIR) if pdf.is_relative_to(INPUT_DIR) else pdf.name
        print(f"[{index}/{len(pdfs)}] {rel}")
        try:
            results.append(process_pdf(pdf))
        except Exception as exc:
            print("  ERROR:", exc)
            try:
                torch.cuda.empty_cache()
            except Exception:
                pass
            gc.collect()
            results.append(
                {"doc": safe_doc_id(pdf), "status": "error", "error": str(exc)}
            )

    summary = {
        "model": MODEL_ID,
        "dtype": str(DTYPE).replace("torch.", ""),
        "seconds": round(time.time() - batch_started, 1),
        "total": len(results),
        "ok": sum(1 for r in results if r.get("status") == "ok"),
        "failed": sum(1 for r in results if r.get("status") == "error"),
        "skipped": sum(1 for r in results if r.get("status") == "skipped"),
        "results": results,
    }
    (OUTPUT_DIR / "run_summary.json").write_text(
        json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    print(
        f"\nDone in {summary['seconds']}s: "
        f"{summary['ok']} ok, {summary['skipped']} skipped, {summary['failed']} failed"
    )
    print("Results:", OUTPUT_DIR)
    return results


In [ ]:
#@title 6. Wait for PDFs, then process
import time

print("=" * 72)
print("WAITING FOR PDFs")
print("Upload to:", INPUT_DIR)
print("In Colab: left sidebar -> folder icon -> open that folder -> upload PDFs")
print("=" * 72)

deadline = time.time() + WAIT_MINUTES * 60
last_note = 0.0
pdfs = list_pdfs()

while not pdfs and time.time() < deadline:
    if time.time() - last_note > 30:
        print("  still waiting ...", time.strftime("%H:%M:%S"))
        last_note = time.time()
    time.sleep(5)
    pdfs = list_pdfs()

if not pdfs:
    print("No PDFs found. Upload them to the input folder and re-run this cell.")
else:
    print(f"Found {len(pdfs)} PDF(s). Starting ...")
    run_all()
    print("\nAdd more PDFs and re-run THIS cell to process them.")


WAITING FOR PDFs
Upload to: /content/input
In Colab: left sidebar -> folder icon -> open that folder -> upload PDFs
Found 1 PDF(s). Starting ...
Found 1 PDF(s)
[1/1] FTCCI Panel A All.pdf
  rendered 26/26 page(s) at 200 dpi
  chunk 1/13: page_0001 .. page_0002
    running model (this can take several minutes) ...
[PageRope layer=0 row0] n_pages=2 page_mod=2 folded_pages=[0, 1, 0] prompt(page0)[0-4#5] uniq_coords=553/553 p1:img#273 txt#0 ok=False p2:img#273 txt#0 ok=False
[PageAwareKL layer=0] image_page_mode=marker page_token_id=128827 prompt_page_markers=2 per_page_image_tokens=273 page_markers=2 image_tokens=546 query_pages=2 image_pages=2 page_aware_enabled=True
[PageAwareKL layer=0 row0] markers@[1, 275] p0:img[2-274#273] pre[1-274#274] txt[none] ok=False p1:img[276-548#273] pre[275-548#274] txt[none] ok=False


/usr/local/lib/python3.13/dist-packages/triton/language/core.py:2625: UserWarning: tl.make_block_ptr is deprecated. Use TensorDescriptor or tl.make_tensor_descriptor instead.
  warn("tl.make_block_ptr is deprecated. Use TensorDescriptor or tl.make_tensor_descriptor instead.")


[DASH] Triton block-sparse path active (S=553, block_size=32, remote_topk=12, local_window=128)
[PageAwareKL layer=1] image_page_mode=marker page_token_id=128827 prompt_page_markers=2 per_page_image_tokens=273 page_markers=2 image_tokens=546 query_pages=2 image_pages=2 page_aware_enabled=True
[PageAwareKL layer=1 row0] markers@[1, 275] p0:img[2-274#273] pre[1-274#274] txt[none] ok=False p1:img[276-548#273] pre[275-548#274] txt[none] ok=False
[DASH] Triton block-sparse path active (S=553, block_size=32, remote_topk=12, local_window=128)
[PageAwareKL layer=2] image_page_mode=marker page_token_id=128827 prompt_page_markers=2 per_page_image_tokens=273 page_markers=2 image_tokens=546 query_pages=2 image_pages=2 page_aware_enabled=True
[PageAwareKL layer=2 row0] markers@[1, 275] p0:img[2-274#273] pre[1-274#274] txt[none] ok=False p1:img[276-548#273] pre[275-548#274] txt[none] ok=False
[DASH] Triton block-sparse path active (S=553, block_size=32, remote_topk=12, local_window=128)
[PageAwareKL

In [ ]:
#@title 7. Preview the Markdown
from IPython.display import Markdown, display

md_files = sorted(OUTPUT_DIR.glob("*.md"))
if not md_files:
    print("No Markdown files yet.")
for md_file in md_files:
    print("\n" + "=" * 72)
    print(md_file)
    print("=" * 72)
    text = md_file.read_text(encoding="utf-8", errors="replace")
    display(Markdown(text[:6000]))
    if len(text) > 6000:
        print(f"... preview truncated; open {md_file.name} for the full text.")



/content/output/FTCCI_Panel_A_All.md


The Federation of Telangana Chambers of Commerce and Industry

Members' Directory 2025

<table><tr><td>Panel No</td><td>Name &amp; Address</td><td>Panel No</td><td>Name &amp; Address</td></tr><tr><td>1</td><td>BIRLANU PVT. LTD.(Formerly HIL LTD.)Office No 1 &amp; 2, Level 7,SLN Terminus,Sy.No 133, 7th Floor,Near Botanical Gardens,Gachibowli,Hyderabad 500032</td><td></td><td>mahipalreddy.saddi@biologicale.com;eswarareddy.n@biologicale.com;vanisri.koneru@biologicale.com</td></tr><tr><td>PhMail</td><td>68249000customer.care@birlanu.com;bnduth.sripada@birlanu.com</td><td>WebFax</td><td>www.biologicale.comAzamabad, Patancheru,Gaganpahad, Shameerpet,Dehradun</td></tr><tr><td>WebFact</td><td>www.birlanu.comHyderabad (AP), Faridabad(Haryana), Dharuhera (Haryana)Thrissor, Satharia(U.P.), Jasidih(Jharkhand),Wada(Maharashtra) Timmapur (A.P)Kondapally (A.P) Chennai</td><td>BrRep.</td><td>Das Chambers, Mumbai.S. Mahipal Reddy - CSN. Eswara ReddySr. Vice President-Legal</td></tr><tr><td>BrRep.NB</td><td>New Plant at Balasore(Akshat Seth - Mg.Dir.Manufacturer and Exporters ofFibre Cement Sheets,Accessories and Fittings, AACBlocks, Wall Panels</td><td>5</td><td>Mumbai of India</td></tr><tr><td>3</td><td>AUTOMOTIVEMANUFACTURERS PVT. LTD.8571, R.P.RoadSecunderabad 500 003</td><td>PhMail</td><td>2776-103/1065, AzamabadHyderabad 500 020</td></tr><tr><td>PhMail</td><td>27543/701/5454traglycer@automotivelm.com;harshmehta@automotivelm.com;klp@automotivelm.com;kshiva.sec@automotivelm.com</td><td>WebFact</td><td>27868000corporate@vsstind.com;sanjay.wali@vsstind.com;anish@vsstind.com;panimangipudi@vsstind.com;rekha@vsstind.comwww.vsthyd.com</td></tr><tr><td>WebFact</td><td>www.automotivelm.comS.No.73, Jeedimetila (V),R.R.Dist</td><td>WebFact</td><td>www.vsthyd.comSurvey No.288/289,Muppiredpdally, Toopran,Medchal-Malkajgiri</td></tr><tr><td>Br</td><td>Hyderabad, Secunderabad,Madhurai, Chennai, Guntur,Nizamabad, Ongole, Warangal,Karimnagar, Bangalore</td><td>BrRep.</td><td>Anish Gupta - CFO27688226Phani K. Mangipudi - CS27688437</td></tr><tr><td>Rep.</td><td>T.R.Ganesh Aiyer - PresidentHarsh B. MehtaAsso. VP - Special Assignment9963391962K. Lakshmipathi RaoGM-Accounts9618904545</td><td>NB</td><td>Manufacture &amp; Marketing ofCigarettes and UnmanufacturedTobacco</td></tr><tr><td>NB</td><td>Exclusive Automobile dealersfor Ashok Leyland, KIA, MGM,Kobelco, and Plagg</td><td>6</td><td>GOCL CORPORATIONPVT. LTD.FCC6+PF9, Corporate Office,IDL Access Road, Kukatpally,Hyderabad 500072</td></tr><tr><td>4</td><td>BIOLOGICAL E. LTD.Mr.S.Mahipal Reddy,Company Secretary18/1 &amp; 3, AzambabadHyderabad 500 020</td><td>PhMobMailRep:</td><td>2381067119948687411secretarial@gocclcorp.com;ravijain@gocclcorp.com;mamruthrao@gocclcorp.comwww.gocclcorp.com</td></tr><tr><td>PhMail</td><td>68274166info@biologicale.com;</td><td>WebFact</td><td>Hyderabad, Rourkela,Pashamylaram, and SilvassaMumbai, Kolkata, Bangalore,Vizag, Singrauli &amp; Delhi</td></tr></table>

<table><tr><td>Panel No</td><td>Name &amp; Address</td></tr><tr><td>NB</td><td>Manufacturing, Marketing, and Export of Industrial Explosives, Detonating Fuses, and Detonators, and also undertakes Infrastructure Projects.</td></tr><tr><td>8</td><td>USHA INTERNATIONAL LTD.(Hyderabad Engineering Inds.)P.O.Balanagar Township,Balanagar, Hyderabad 500037</td></tr><tr><td>Ph</td><td>23078691/23079197</td></tr><tr><td>Mob</td><td>9160016196</td></tr><tr><td>Mail</td><td>basu_santanu@usha.com;bensam_raja@usha.com;mahendarreddy_bodanapu@usha.com;Pulla_Srinivas@usha.com</td></tr><tr><td>Web</td><td>www.usha.com</td></tr><tr><td>Rep.</td><td>Santanu BasuVice President &amp; Unit HeadBensam K Raja - GM HRVishal Karan -AYP</td></tr><tr><td>NB</td><td>Manufacturers &amp; Exporters ofFuel Injection Equipment andPrecision AutomotiveComponents</td></tr><tr><td>16</td><td>COROMANDELINTERNATIONAL LTD.1-2-10 Coromandel HouseSardar Patel RoadSecunderabad 500 003</td></tr><tr><td>Ph</td><td>27842034/27847212</td></tr><tr><td>Mail</td><td>cfl.sec@gnhyd.global.net.in;sarvamangaladevib@coromandel.murugappa.com</td></tr><tr><td>Web</td><td>www.cflindia.com</td></tr><tr><td>Fact</td><td>Sriharipuram, Malikapuram P.O,Visakhapatnam</td></tr><tr><td>Rep.</td><td>Amitab MisraCorporate HR ManagerP.Nagarajan - VP (Fin.&amp;Admn)Arun Leslie George - GM-HR</td></tr><tr><td>NB</td><td>Manufacturers Phosphatfertilisers under the brand&quot;Gromor&quot;</td></tr><tr><td>21</td><td>AGI GREENPAC PVT. LTD.(An SBU of HSIL Ltd.)_Glass Factory Road,Off Motinagar, P.B.No.1930,Sanathnagar P.O.,Hyderabad 500 018</td></tr><tr><td>Ph</td><td>23831774</td></tr><tr><td>Mob</td><td>9849606162 / 9491061204</td></tr><tr><td>Mail</td><td>secretary-hr@agi-glaspac.com;govind@agi-glaspac.com;rk@agi-glaspac.com;oppanday@agi-glaspac.com;nmrao.i@agi-glaspac</td></tr></table>

(Associate - City)

GENEAL

FTCCI Members' Directory 2025

3

(Associates - City)

ANEAL

The Federation of Telangana Chambers of Commerce and Industry

Members' Directory 2025

<table><tr><td>Panel No</td><td>Name &amp; Address</td><td>Panel No</td><td>Name &amp; Address</td><td>Panel No</td><td>Name &amp; Address</td></tr><tr><td>Web Fact Rep.</td><td>www.agi-glaspac.comHyderabad, BhongirRajesh KkhoslaPresident &amp; CEO9810530695Govind Jee RaiSr. VP (P&amp;A)HR9849606162Om Prakash Pandey - CFO9849215268</td><td>Mail</td><td>sailajach@margardasi.com;corporatesaccounts@margardasi.com;svswamy@margardasi.com;rajajl_p@margardasi.com</td><td>NB</td><td>Manufacturers of Cigarettes,Unmanufactured Tobacco,Confectionery, Cigars and Tea</td></tr><tr><td>NB</td><td>Manufacturers of Glass Bottles/Pet Bottles / Clozures</td><td>Web</td><td>mavw.margardasi.com</td><td>38</td><td>NAVALTD.Nava Bharat Chambers,6-3-1109/1, 3rd Floor,Rajbhavan RoadHyderabad 500 082</td></tr><tr><td>25</td><td>MAHINDRA &amp; MAHINDRA LTD.Automotive DivisionMahindra Nagar, ZaheerabadSangareddy District 502 22008451-284241/284244</td><td>34</td><td>ALKALI METALS LTD.B-5, Block III, IDA UppalHyderabad 500 039<

... preview truncated; open FTCCI_Panel_A_All.md for the full text.


In [ ]:
#@title 8. Download all results as a ZIP
if AUTO_DOWNLOAD_ZIP and any(OUTPUT_DIR.glob("*.md")):
    import datetime
    stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    zip_base = Path("/content") / f"unlimited_ocr_output_{stamp}"
    zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=str(OUTPUT_DIR))
    print("ZIP:", zip_path)
    try:
        from google.colab import files
        files.download(zip_path)
    except Exception as exc:
        print("Automatic download failed; use the file pane:", exc)
else:
    print("Nothing to download yet.")


ZIP: /content/unlimited_ocr_output_20261107_134442.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Troubleshooting

**"No GPU detected"**
Use `Runtime -> Change runtime type -> Hardware accelerator -> GPU`.

**Transformers / import errors after installing**
Use `Runtime -> Restart runtime`, then `Runtime -> Run all`. This notebook pins
`transformers==4.52.1`, which is the version the model's remote code targets.

**CUDA out of memory**
- Set `CHUNK_PAGES = 1`
- Set `MODE = "base"`
- Lower `MAX_TOKENS_PER_PAGE` (e.g. 4000)
- Use a bigger GPU or a high-RAM runtime
The notebook also retries automatically by splitting a request into smaller
page groups when it hits OOM.

**T4 / BF16 warning**
T4 (Turing) has no native BF16, so the notebook uses float16 and patches the
model's image preprocessing. An L4/A100 is faster and more accurate.

**Triton / custom kernel errors**
Set `USE_TRITON = False` in the configuration cell, use
`Runtime -> Restart runtime`, then run all. This uses the slower pure-PyTorch
attention path.

**The output is cut off**
`name.meta.json` will have `"truncated": true`. Raise `MAX_TOKENS_PER_PAGE`, or
process fewer pages per request with `CHUNK_PAGES`.

**Colab disconnected and files are gone**
Set `USE_DRIVE = True` in the configuration cell, or download the ZIP before
the runtime ends. Colab's `/content` storage is temporary.

**Long documents**
A free Colab GPU is not meant for hundred-page documents. Use `CHUNK_PAGES` to
limit memory, and expect long runtimes. For production, use the project's
SGLang server (`start_server.sh` + `run_ocr.py`).
